# DeepMzyme Colab Training Notebook

This notebook runs DeepMzyme training from Google Colab using the existing command-line interface in `src/train.py`. It is designed for baseline-first experiments on the trusted non-overlapped PinMyMetal split, with clear controls for task, model preset, hyperparameters, output copying, and run summarization.

The notebook does not duplicate training logic. It builds a CLI command, runs it only when you explicitly enable training, then calls `src/report_runs.py` to create a comparison CSV and optional figure.

## 1. Mount Google Drive

Use Drive for the compressed Colab bundle and for long-term run outputs. Runtime-local paths under `/content` are faster but disappear when the Colab session ends.

In [ ]:
#@title Mount Google Drive
MOUNT_DRIVE = True  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive mount skipped. Make sure bundle and output paths are accessible.')

## 2. Configure Repository, Bundle, and Output Paths

Edit this cell first. The notebook always clones DeepMzyme from Git into `REPO_DIR`, so set `REPO_GIT_URL` and `REPO_GIT_BRANCH` before running the dependency cell.

`DRIVE_DATA_DIR` is the persistent Google Drive `.data` directory. The notebook reads the compressed bundle from `DRIVE_DATA_DIR/Colab_Bundles`, unpacks it into runtime-local `/content`, trains into runtime-local `/content`, then copies final run outputs back to `DRIVE_DATA_DIR/Train_Parameters_Models_and_Results`.

In [ ]:
#@title Paths and repository setup
from pathlib import Path

REPO_DIR = '/content/DeepMzyme'  #@param {type:"string"}
REPO_GIT_URL = 'https://github.com/MECHTI1/DeepMzyme.git'  #@param {type:"string"}
REPO_GIT_BRANCH = 'main'  #@param {type:"string"}

DRIVE_DATA_DIR = '/content/drive/MyDrive/DeepMzyme/.data'  #@param {type:"string"}
BUNDLE_FILENAME = 'train_and_test_sets_structures_non_overlapped_pinmymetal_colab_bundle_structures_validated_fast.tar.zst'  #@param {type:"string"}
UNPACK_DIR = '/content/deepmzyme_bundle'  #@param {type:"string"}
DATASET_NAME = 'train_and_test_sets_structures_non_overlapped_pinmymetal'  #@param {type:"string"}

LOCAL_RUNS_DIR = '/content/deepmzyme_runs'  #@param {type:"string"}
DRIVE_OUTPUT_SUBDIR = 'Train_Parameters_Models_and_Results'  #@param {type:"string"}

repo_dir = Path(REPO_DIR).expanduser().resolve()
drive_data_dir = Path(DRIVE_DATA_DIR).expanduser()
BUNDLE_PATH = str(drive_data_dir / 'Colab_Bundles' / BUNDLE_FILENAME)
bundle_path = Path(BUNDLE_PATH).expanduser()
unpack_dir = Path(UNPACK_DIR).expanduser().resolve()
local_runs_dir = Path(LOCAL_RUNS_DIR).expanduser().resolve()
drive_output_dir = drive_data_dir / DRIVE_OUTPUT_SUBDIR

print('Repository:', repo_dir)
print('Drive .data directory:', drive_data_dir)
print('Bundle:', bundle_path)
print('Unpack directory:', unpack_dir)
print('Local runs directory:', local_runs_dir)
print('Drive output directory:', drive_output_dir)

## 3. Install and Check Dependencies

This cell prepares the repository and checks core imports. Installing dependencies can take several minutes in a fresh Colab runtime. If you already prepared the runtime, set `INSTALL_REQUIREMENTS` to `False` and keep the import check enabled.

In [ ]:
#@title Repository and dependency checks
import os
import subprocess
import sys
from pathlib import Path

INSTALL_REQUIREMENTS = True  #@param {type:"boolean"}
CHECK_IMPORTS = True  #@param {type:"boolean"}

if not REPO_GIT_URL or '<' in REPO_GIT_URL:
    raise ValueError('Set REPO_GIT_URL to your DeepMzyme repository URL before cloning.')
if repo_dir.exists():
    print(f'Repository already exists at {repo_dir}; skipping clone.')
else:
    subprocess.run(['git', 'clone', '--branch', REPO_GIT_BRANCH, REPO_GIT_URL, str(repo_dir)], check=True)

requirements_path = repo_dir / 'src' / 'requirements.txt'
if INSTALL_REQUIREMENTS:
    if requirements_path.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
    else:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install',
            'torch-geometric==2.7.0', 'biopython==1.87', 'biotite>=1.6', 'propka>=3.5', 'gemmi>=0.7', 'pandas', 'matplotlib'
        ], check=True)

if CHECK_IMPORTS:
    src_path = str(repo_dir / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    import torch
    import pandas as pd
    import torch_geometric
    print('Python:', sys.executable)
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('PyTorch Geometric:', torch_geometric.__version__)

## 4. Unpack the Colab Bundle

This extracts the training/test structures and CSV files into `UNPACK_DIR`. The notebook keeps extracted data outside the repository `.data` tree so smoke or setup work does not modify project data.

In [ ]:
#@title Unpack bundle
import shutil
import subprocess
from pathlib import Path

FORCE_REUNPACK = False  #@param {type:"boolean"}

def run_checked(cmd):
    print('+', ' '.join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

def unpack_archive(source: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(source.suffixes).lower()
    if suffixes.endswith('.tar.zst'):
        if shutil.which('zstd') is None:
            run_checked(['apt-get', 'update'])
            run_checked(['apt-get', 'install', '-y', 'zstd'])
        run_checked(['tar', '--use-compress-program=zstd', '-xf', source, '-C', destination])
    elif suffixes.endswith(('.tar.gz', '.tgz', '.tar')):
        run_checked(['tar', '-xf', source, '-C', destination])
    else:
        raise ValueError(f'Unsupported bundle archive type: {source}')

if FORCE_REUNPACK and unpack_dir.exists():
    shutil.rmtree(unpack_dir)

if bundle_path.is_dir():
    unpack_dir = bundle_path.resolve()
    print(f'Using already-unpacked bundle directory: {unpack_dir}')
else:
    if not bundle_path.exists():
        raise FileNotFoundError(f'Bundle not found: {bundle_path}')
    marker = unpack_dir / '.deepmzyme_bundle_unpacked'
    if marker.exists() and not FORCE_REUNPACK:
        print(f'Bundle already unpacked at {unpack_dir}. Set FORCE_REUNPACK=True to extract again.')
    else:
        unpack_archive(bundle_path, unpack_dir)
        marker.write_text('ok\n', encoding='utf-8')
        print(f'Unpacked bundle into {unpack_dir}')

## 5. Verify Train/Test Directories and Summary CSVs

DeepMzyme training needs a train structure directory, a test structure directory, and MAHOMES-style summary CSVs with site-level columns such as PDB ID, EC number, metal residue number, and metal residue type. The bundle may also contain structure-level CSV artifacts; those are useful metadata, but they are not used here unless they match the training loader requirements.

In [ ]:
#@title Detect and verify dataset paths
import csv
from pathlib import Path

TRAIN_DIR_OVERRIDE = ''  #@param {type:"string"}
TEST_DIR_OVERRIDE = ''  #@param {type:"string"}
TRAIN_CSV_OVERRIDE = ''  #@param {type:"string"}
TEST_CSV_OVERRIDE = ''  #@param {type:"string"}

REQUIRED_SUMMARY_ALIASES = {
    'pdbid': {'pdbid', 'structure'},
    'metal residue number': {'metal residue number', 'chain_resi'},
    'EC number': {'ec number', 'ecnumber'},
    'metal residue type': {'metal residue type', 'metaltype'},
}

def has_required_summary_columns(path: Path) -> bool:
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            fields = {field.strip().lower() for field in (reader.fieldnames or []) if field}
    except Exception:
        return False
    return all(fields.intersection(aliases) for aliases in REQUIRED_SUMMARY_ALIASES.values())

def find_dataset_root(root: Path, dataset_name: str) -> Path:
    candidates = [
        root / '.data' / dataset_name,
        root / dataset_name,
        root,
    ]
    candidates.extend(path for path in root.rglob(dataset_name) if path.is_dir())
    for candidate in candidates:
        if (candidate / 'train').is_dir() and (candidate / 'test').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find train/test directories for dataset {dataset_name!r} under {root}')

def find_summary_csv(split_dir: Path, split_name: str) -> Path:
    preferred_names = [
        'catalytic_only_summary.csv',
        'final_data_summarazing_table_transition_metals_only_catalytic.csv',
        f'{DATASET_NAME}_{split_name}.csv',
    ]
    candidates = [split_dir / name for name in preferred_names]
    candidates.extend(sorted(split_dir.glob('*.csv')))
    candidates.extend(sorted((unpack_dir / '.data' / 'Colab_Bundles').glob(f'**/*{split_name}*.csv')))
    for candidate in candidates:
        if candidate.exists() and has_required_summary_columns(candidate):
            return candidate.resolve()
    found = [str(path) for path in candidates if path.exists()]
    raise FileNotFoundError(
        f'No MAHOMES-style {split_name} summary CSV found. Checked: {found}. '
        'If your bundle only contains structure_name/ec_numbers/metal_type CSVs, rebuild it with the site-level summary CSV included.'
    )

dataset_root = find_dataset_root(unpack_dir, DATASET_NAME)
train_dir = Path(TRAIN_DIR_OVERRIDE).expanduser().resolve() if TRAIN_DIR_OVERRIDE else dataset_root / 'train'
test_dir = Path(TEST_DIR_OVERRIDE).expanduser().resolve() if TEST_DIR_OVERRIDE else dataset_root / 'test'
train_csv = Path(TRAIN_CSV_OVERRIDE).expanduser().resolve() if TRAIN_CSV_OVERRIDE else find_summary_csv(train_dir, 'train')
test_csv = Path(TEST_CSV_OVERRIDE).expanduser().resolve() if TEST_CSV_OVERRIDE else find_summary_csv(test_dir, 'test')

for label, path in [('train_dir', train_dir), ('test_dir', test_dir), ('train_csv', train_csv), ('test_csv', test_csv)]:
    if not path.exists():
        raise FileNotFoundError(f'{label} does not exist: {path}')

train_structures = sorted(list(train_dir.glob('*.pdb')) + list(train_dir.glob('*.cif')) + list(train_dir.glob('*.mmcif')))
test_structures = sorted(list(test_dir.glob('*.pdb')) + list(test_dir.glob('*.cif')) + list(test_dir.glob('*.mmcif')))
if not train_structures or not test_structures:
    raise ValueError(f'Expected structure files in both train and test directories: {train_dir}, {test_dir}')

print('Dataset root:', dataset_root)
print('Train structures:', len(train_structures), train_dir)
print('Test structures:', len(test_structures), test_dir)
print('Train summary CSV:', train_csv)
print('Test summary CSV:', test_csv)

## 6. Select Task, Model Preset, and Hyperparameters

Baseline-first order is: Only-GVP, Only-ESM, GVP + late fusion, then GVP + early fusion. The metal collapsed-4 option uses the current CLI support for collapsed-4 validation/test metrics; it does not create a separate 4-class training head.

In [ ]:
#@title Training configuration
from datetime import datetime

TASK_MODE = 'metal_6_class'  #@param ['metal_6_class', 'metal_collapsed4_metric', 'ec_prediction']
MODEL_PRESET = 'Only-GVP'  #@param ['Only-GVP', 'Only-ESM', 'GVP + late fusion', 'GVP + early fusion']

EPOCHS = 50  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
LEARNING_RATE = 3e-4  #@param {type:"number"}
WEIGHT_DECAY = 1e-4  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
VAL_FRACTION = 0.15  #@param {type:"number"}
DEVICE = 'cuda'  #@param ['cuda', 'cpu']
RUN_NAME = ''  #@param {type:"string"}

EC_LABEL_DEPTH = 1  #@param {type:"integer"}
EC_CONTRASTIVE_WEIGHT = 0.0  #@param {type:"number"}
NODE_FEATURE_SET = 'conservative'  #@param ['conservative', 'expanded']
SPLIT_BY = 'pdbid'  #@param ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id']

ALLOW_MISSING_ESM_EMBEDDINGS = True  #@param {type:"boolean"}
PREPARE_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_EXTERNAL_FEATURES = True  #@param {type:"boolean"}
RUN_HELD_OUT_TEST_EVAL = True  #@param {type:"boolean"}

task_map = {
    'metal_6_class': ('metal', 'val_metal_balanced_acc'),
    'metal_collapsed4_metric': ('metal', 'val_metal_collapsed4_balanced_acc'),
    'ec_prediction': ('ec', 'val_ec_balanced_acc'),
}
preset_map = {
    'Only-GVP': {'model_architecture': 'only_gvp'},
    'Only-ESM': {'model_architecture': 'only_esm'},
    'GVP + late fusion': {'model_architecture': 'gvp', 'fusion_mode': 'late_fusion'},
    'GVP + early fusion': {'model_architecture': 'gvp', 'fusion_mode': 'early_fusion', 'early_esm_dim': 32, 'early_esm_dropout': 0.2},
}

task_name, selection_metric = task_map[TASK_MODE]
preset = preset_map[MODEL_PRESET]
safe_preset_name = MODEL_PRESET.lower().replace(' + ', '_').replace('-', '_').replace(' ', '_')
if not RUN_NAME:
    RUN_NAME = f'{TASK_MODE}_{safe_preset_name}_seed{SEED}'

target_run_dir = local_runs_dir / RUN_NAME
if target_run_dir.exists():
    RUN_NAME = f'{RUN_NAME}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    target_run_dir = local_runs_dir / RUN_NAME

print('Task:', task_name)
print('Selection metric:', selection_metric)
print('Model preset:', MODEL_PRESET)
print('Run name:', RUN_NAME)
print('Expected run directory:', target_run_dir)

## 7. Build the Training Command

Review the command before running. The held-out test set is used only for final reporting of the selected checkpoint; validation metrics control checkpoint selection.

In [ ]:
import os
import shlex
import sys
from pathlib import Path

local_runs_dir.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, str(repo_dir / 'src' / 'train.py'),
    '--task', task_name,
    '--structure-dir', str(train_dir),
    '--summary-csv', str(train_csv),
    '--runs-dir', str(local_runs_dir),
    '--run-name', RUN_NAME,
    '--model-architecture', preset['model_architecture'],
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--weight-decay', str(WEIGHT_DECAY),
    '--seed', str(SEED),
    '--val-fraction', str(VAL_FRACTION),
    '--device', DEVICE,
    '--split-by', SPLIT_BY,
    '--selection-metric', selection_metric,
    '--node-feature-set', NODE_FEATURE_SET,
]

if 'fusion_mode' in preset:
    train_cmd.extend(['--fusion-mode', preset['fusion_mode']])
if preset.get('fusion_mode') == 'early_fusion':
    train_cmd.extend(['--early-esm-dim', str(preset['early_esm_dim']), '--early-esm-dropout', str(preset['early_esm_dropout'])])
if task_name == 'ec':
    train_cmd.extend(['--ec-label-depth', str(EC_LABEL_DEPTH)])
    train_cmd.extend(['--ec-contrastive-weight', str(EC_CONTRASTIVE_WEIGHT)])
if RUN_HELD_OUT_TEST_EVAL:
    train_cmd.extend(['--test-structure-dir', str(test_dir), '--test-summary-csv', str(test_csv), '--run-test-eval'])
if ALLOW_MISSING_ESM_EMBEDDINGS:
    train_cmd.append('--allow-missing-esm-embeddings')
if not PREPARE_MISSING_ESM_EMBEDDINGS:
    train_cmd.append('--no-prepare-missing-esm-embeddings')
if ALLOW_MISSING_EXTERNAL_FEATURES:
    train_cmd.append('--allow-missing-external-features')

env = os.environ.copy()
env['PYTHONPATH'] = str(repo_dir / 'src') + os.pathsep + env.get('PYTHONPATH', '')

print('Training command:')
print(' '.join(shlex.quote(str(part)) for part in train_cmd))

## 8. Run Training

This is the only cell that starts training. Keep `RUN_TRAINING` set to `False` until paths, task, model preset, and hyperparameters have been checked.

In [ ]:
#@title Execute training command
RUN_TRAINING = False  #@param {type:"boolean"}

if not RUN_TRAINING:
    print('Training skipped. Set RUN_TRAINING=True when you are ready.')
else:
    if DEVICE == 'cuda':
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError('DEVICE is cuda, but torch.cuda.is_available() is False. Change DEVICE to cpu or enable a GPU runtime.')
    subprocess.run(train_cmd, cwd=str(repo_dir), env=env, check=True)
    print('Training completed. Run directory:', target_run_dir)

## 9. Copy Run Outputs Back to Drive

Copy the local run directory to Drive after training. If `LOCAL_RUNS_DIR` already points inside Drive, this still creates a stable copy under `DRIVE_DATA_DIR/Train_Parameters_Models_and_Results/runs` by default.

In [ ]:
#@title Copy outputs to Drive
COPY_OUTPUTS_TO_DRIVE = True  #@param {type:"boolean"}

import shutil

drive_runs_dir = drive_output_dir / 'runs'
drive_run_dir = drive_runs_dir / RUN_NAME

if COPY_OUTPUTS_TO_DRIVE:
    if not target_run_dir.exists():
        print(f'Run directory does not exist yet: {target_run_dir}')
    else:
        drive_runs_dir.mkdir(parents=True, exist_ok=True)
        if drive_run_dir.exists():
            shutil.rmtree(drive_run_dir)
        shutil.copytree(target_run_dir, drive_run_dir)
        print('Copied run output to:', drive_run_dir)
else:
    print('Copy skipped.')

## 10. Summarize Runs with `src/report_runs.py`

The summary CSV is the main comparison table. The optional figure is created only when matplotlib is available and the selected validation metric is numeric.

In [ ]:
#@title Generate run summary
SUMMARY_SOURCE = 'drive_outputs'  #@param ['local_runs', 'drive_outputs']

import subprocess
from pathlib import Path

summary_runs_dir = local_runs_dir if SUMMARY_SOURCE == 'local_runs' else drive_runs_dir
summary_csv = drive_output_dir / 'baseline_first_summary.csv'
summary_figure = drive_output_dir / 'baseline_first_summary.png'
drive_output_dir.mkdir(parents=True, exist_ok=True)

report_cmd = [
    sys.executable, str(repo_dir / 'src' / 'report_runs.py'),
    '--runs-dir', str(summary_runs_dir),
    '--out-csv', str(summary_csv),
    '--out-figure', str(summary_figure),
]
print('Report command:')
print(' '.join(shlex.quote(str(part)) for part in report_cmd))

if not summary_runs_dir.exists():
    print(f'No run directory found yet: {summary_runs_dir}')
else:
    subprocess.run(report_cmd, cwd=str(repo_dir), env=env, check=True)
    print('Summary CSV:', summary_csv)
    print('Summary figure:', summary_figure)

## 11. Display the Summary Table and Figure

Use this table to compare runs by validation-selected metrics first. Test metrics are for final held-out reporting after selecting checkpoints by validation performance.

In [ ]:
from IPython.display import Image, display
import pandas as pd

if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    display(summary_df)
else:
    print(f'Summary CSV not found yet: {summary_csv}')

if summary_figure.exists():
    display(Image(filename=str(summary_figure)))
else:
    print(f'Optional summary figure not found: {summary_figure}')

## 12. Final Model-Selection Checklist

- Confirm every final run used the non-overlapped PinMyMetal split, or clearly label exact/possibly-overlapped runs as secondary reference only.
- Compare Only-GVP, Only-ESM, GVP + late fusion, and GVP + early fusion before moving to complex fusion modes.
- Select checkpoints and model variants by validation metrics, not by repeatedly checking the held-out test set.
- For metal prediction, report both 6-class metrics and collapsed-4 metrics from the selected checkpoint.
- For EC prediction, report supported EC level metrics from the selected checkpoint.
- Preserve `run_config.json`, `run_metadata.json`, `dataset_summary.json`, `test_report.json`, selected checkpoint path, split identity, overlap status, and the summary CSV/figure in Drive.